# Logit Lens — Basic Exploration

**What is the Logit Lens?**

The Logit Lens is a mechanistic interpretability technique that peeks inside a transformer at every layer and asks:
*"If the model had to predict the next token right now, what would it say?"*

### Core idea
A transformer's **residual stream** accumulates information as it flows through layers.
At any intermediate layer `l`, we can apply the **final LayerNorm + unembedding matrix**
directly to the residual stream `h_l` to decode what the model is "thinking" at that depth:

```
h_l  →  ln_final(h_l)  →  unembed(·)  →  softmax  →  top token
```

### Why is this useful?
- Reveals how predictions **form progressively** across layers.
- Shows when the model commits to the correct answer vs. an incorrect one.
- Helps locate **which layers** handle syntax, semantics, and factual recall.

In [1]:
import torch
from transformer_lens.model_bridge import TransformerBridge

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [3]:
# Load model
print("Loading via TransformerBridge ...")
model = TransformerBridge.boot_transformers("SecCoderX/Qwen2.5_Coder_7B_SecCoderX_aligned")
model.eval()

print(f"Model: {model.cfg.model_name}  |  Layers: {model.cfg.n_layers}  |  d_model: {model.cfg.d_model}")
print()

Loading via TransformerBridge ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model: SecCoderX/Qwen2.5_Coder_7B_SecCoderX_aligned  |  Layers: 28  |  d_model: 3584



## Step 1 — Run the Model and Cache All Activations

`model.run_with_cache(prompt)` runs a normal forward pass and returns:
- **`logits`** — final output logits, shape `[batch, seq_len, vocab_size]`
- **`cache`** — an `ActivationCache` with every named intermediate activation, shape `[batch, seq_len, d_model]`

For the Logit Lens we use `blocks.{layer}.hook_resid_post` — the residual stream **after** each complete transformer block (attention + MLP combined).

In [4]:
# Run the model and cache ALL intermediate activations
prompt = "The capital of France is"
logits, cache = model.run_with_cache(prompt)

## Step 2 — Inspect the Cache

The printed cache lists every hook point recorded during the forward pass.
Key entries per layer `N`:

| Hook name | What it captures |
|-----------|------------------|
| `blocks.N.hook_resid_pre` | Residual stream **entering** block N |
| `blocks.N.hook_resid_mid` | After attention, before MLP |
| `blocks.N.hook_resid_post` | After full block N ← **used by Logit Lens** |
| `blocks.N.attn.hook_z` | Per-head attention output vectors |
| `blocks.N.mlp.hook_post` | MLP hidden state after activation function |

## Step 3 — Apply Logit Lens at Each Layer

For each layer `l` (0 → 35), the loop does:

1. **Extract** `h_l = cache["blocks.{l}.hook_resid_post"]` — shape `[1, seq_len, d_model]`
2. **Normalize** with `model.ln_final(h_l)` — same LayerNorm used before the real output
3. **Unembed** with `model.unembed(h_l_normed)` — projects `d_model → vocab_size`
4. **Decode** top predicted token at **position `-1`** (the last token, which predicts the next word)

> We look at position `-1` because in a causal LM, the last token aggregates all prior context
> and is used to predict the next token — that's where `"France is → ?"` is resolved.

In [5]:
# Apply Logit Lens at each layer
for layer in range(model.cfg.n_layers):
    # 1. Get the residual stream state AFTER this layer
    h_l = cache[f"blocks.{layer}.hook_resid_post"]

    # 2. Apply the final LayerNorm (same one the model uses)
    h_l_normed = model.ln_final(h_l)

    # 3. Apply unembedding matrix → this IS the Logit Lens
    logits_l = model.unembed(h_l_normed)

    # 4. Get prediction for the LAST token position
    probs = torch.softmax(logits_l[0, -1, :], dim=-1)
    top_token_id = torch.argmax(probs).item()
    top_token = model.tokenizer.decode(top_token_id)
    confidence = probs[top_token_id].item()

    print(f"Layer {layer:2d}: {top_token:>15s} ({confidence:.1%})")

Layer  0:            ylko (43.1%)
Layer  1:           aniem (9.5%)
Layer  2:           aniem (28.8%)
Layer  3:           aniem (13.3%)
Layer  4:           theid (28.8%)
Layer  5:           theid (20.2%)
Layer  6:           theid (69.1%)
Layer  7:            acyj (41.7%)
Layer  8:            ylko (59.6%)
Layer  9:            ylko (19.0%)
Layer 10:    ----------</ (30.4%)
Layer 11:            acyj (39.3%)
Layer 12:    ----------</ (47.3%)
Layer 13:            acyj (25.4%)
Layer 14:            acyj (28.6%)
Layer 15:    ----------</ (10.1%)
Layer 16:        ereotype (12.7%)
Layer 17:        ereotype (20.1%)
Layer 18:            iado (8.6%)
Layer 19:        ereotype (7.8%)
Layer 20:      	TokenName (10.7%)
Layer 21:            ____ (27.1%)
Layer 22:            ____ (16.8%)
Layer 23:           Paris (98.3%)
Layer 24:           Paris (99.6%)
Layer 25:           Paris (99.8%)
Layer 26:           Paris (99.7%)
Layer 27:           Paris (67.7%)


## Interpreting the Logit Lens Output

Per-layer top-1 predictions for `"The capital of France is"`:

| Layer range | Top prediction | Interpretation |
|-------------|---------------|----------------|
| 0–9 | `ylko`, `aniem`, `theid` | Early layers — generic/nonsense tokens. The model is processing syntax and low-level features, not yet focusing on next-token prediction. |
| 10–22 | `------</`, `ereotype`, `____` | Mid layers — highly scattered, non-semantic tokens. Information is being aggregated and routed in the residual stream, but factual recall has not yet been decoded into the vocabulary space. |
| 23–27 | `"Paris"` | Late layers — the factual answer abruptly materializes with very high confidence (98%+). The model has successfully retrieved the capital of France and committed to the prediction. |

### Key observations
- **Sudden Transition:** Unlike some models that gradually refine their prediction, this model shows a sharp phase transition at Layer 23, jumping straight to `"Paris"` with 98.3% confidence.
- **Nonsense Early Tokens:** In the first ~22 layers, the Logit Lens decodes bizarre tokens (e.g., `ylko`, `____`). This indicates that intermediate representations in this model (Qwen2.5) are highly abstract and not aligned with the final output embedding space until the very end.
- **Final Layer Calibration:** The confidence for `"Paris"` drops slightly in the very last layer (Layer 27) to 67.7%. This is a common phenomenon where the final LayerNorm and unembedding operations adjust the logits to account for alternative valid continuations (e.g., a space before "Paris" or other grammatical nuances).